<a href="https://colab.research.google.com/github/alifia07nisa/coba/blob/main/Klasifikasi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

🔵Upload File Excel

In [ ]:
from google.colab import files
import pandas as pd

# Upload file Excel
uploaded = files.upload()


Saving Dataset.xlsx to Dataset.xlsx


🔵Baca file Excel ke DataFrame

In [ ]:
# Ganti 'data_perpus.xlsx' sesuai nama file kamu
df = pd.read_excel("Dataset.xlsx")

# Tampilkan 5 baris pertama untuk cek data
df.head()


,Provinsi,Year,Reading Frequency per week,Number of Readings per Quarter,Daily Reading Duration (in minutes)
0,Aceh,2020,4.0,2.0,95.0
1,Aceh,2021,5.5,4.5,103.0
2,Aceh,2022,5.0,5.5,94.3
3,Aceh,2023,5.0,5.5,95.0
4,Bali,2020,4.0,2.5,91.0


🔵Normalisasi Min-Max

In [ ]:
# Pilih kolom yang akan dinormalisasi
cols_to_normalize = [
    'Reading Frequency per week',
    'Number of Readings per Quarter',
    'Daily Reading Duration (in minutes)'
]

# Lakukan normalisasi Min-Max
for col in cols_to_normalize:
    df[col + '_norm'] = (df[col] - df[col].min()) / (df[col].max() - df[col].min())

# Tampilkan hasil beberapa baris pertama
df.head()


,Provinsi,Year,Reading Frequency per week,Number of Readings per Quarter,Daily Reading Duration (in minutes),Reading Frequency per week_norm,Number of Readings per Quarter_norm,Daily Reading Duration (in minutes)_norm
0,Aceh,2020,4.0,2.0,95.0,0.333333,0.090909,0.507246
1,Aceh,2021,5.5,4.5,103.0,0.833333,0.545455,0.623188
2,Aceh,2022,5.0,5.5,94.3,0.666667,0.727273,0.497101
3,Aceh,2023,5.0,5.5,95.0,0.666667,0.727273,0.507246
4,Bali,2020,4.0,2.5,91.0,0.333333,0.181818,0.449275


🔵Hitung Indeks Kegemaran Membaca (IKM)

In [ ]:
# Hitung IKM sebagai rata-rata nilai normalisasi
df['IKM'] = df[[col + '_norm' for col in cols_to_normalize]].mean(axis=1)

# Konversi ke persentase
df['IKM_percent'] = df['IKM'] * 100

# Tampilkan beberapa baris untuk cek
df[['Reading Frequency per week_norm',
    'Number of Readings per Quarter_norm',
    'Daily Reading Duration (in minutes)_norm',
    'IKM',
    'IKM_percent']].head()


,Reading Frequency per week_norm,Number of Readings per Quarter_norm,Daily Reading Duration (in minutes)_norm,IKM,IKM_percent
0,0.333333,0.090909,0.507246,0.310496,31.049627
1,0.833333,0.545455,0.623188,0.667325,66.732543
2,0.666667,0.727273,0.497101,0.630347,63.034695
3,0.666667,0.727273,0.507246,0.633729,63.372859
4,0.333333,0.181818,0.449275,0.321476,32.147563


🔵Kategori

In [ ]:
# Fungsi klasifikasi sesuai rumus Excel
def klasifikasi_ikm_excel(ikm):
    if ikm >= 66:
        return 'Tinggi'
    elif ikm >= 56:
        return 'Sedang'
    else:
        return 'Rendah'

# Terapkan fungsi ke kolom IKM_percent
df['Kategori_IKM'] = df['IKM_percent'].apply(klasifikasi_ikm_excel)

# Cek hasil beberapa baris
df[['IKM_percent', 'Kategori_IKM']].head()


,IKM_percent,Kategori_IKM
0,31.049627,Rendah
1,66.732543,Tinggi
2,63.034695,Sedang
3,63.372859,Sedang
4,32.147563,Rendah


🔵Cek Jumlah Kategori

In [ ]:
# Hitung jumlah tiap kategori
counts = df['Kategori_IKM'].value_counts()

# Tambahkan total
counts['Total'] = counts.sum()

# Tampilkan
counts


,count
Kategori_IKM,
Rendah,71
Sedang,37
Tinggi,32
Total,140


🔵Visualisasi

In [ ]:
!pip install plotly


🔵Pie Chart

In [ ]:
import plotly.express as px

# Buat dataframe untuk chart
chart_data = df['Kategori_IKM'].value_counts().reset_index()
chart_data.columns = ['Kategori', 'Jumlah']

# Buat pie chart
fig = px.pie(chart_data, names='Kategori', values='Jumlah',
             title='Distribusi Kategori IKM Peserta',
             color='Kategori',
             color_discrete_map={'Rendah':'red', 'Sedang':'orange', 'Tinggi':'green'})

fig.show()


🔵Bar Chart

In [ ]:
fig2 = px.bar(chart_data, x='Kategori', y='Jumlah',
              title='Jumlah Peserta per Kategori IKM',
              color='Kategori',
              color_discrete_map={'Rendah':'red', 'Sedang':'orange', 'Tinggi':'green'},
              text='Jumlah')

fig2.show()


🔵Bar Chart per Provinsi

In [ ]:
import plotly.express as px

# Hitung rata-rata IKM per provinsi
provinsi_ikm = df.groupby('Provinsi')['IKM_percent'].mean().reset_index()

# Bar chart
fig = px.bar(
    provinsi_ikm,
    x='Provinsi',
    y='IKM_percent',
    title='Rata-rata Indeks Kegemaran Membaca (IKM) per Provinsi',
    text_auto='.2f'
)

fig.update_layout(xaxis_tickangle=-45)
fig.show()
